# System Dependencies and Installing Python Libraries

In [ ]:
# System tools
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr libtesseract-dev

# Core PDF + OCR
%pip install -q pymupdf pytesseract pillow

# LangChain modern packages
%pip install -q langchain langchain-core langchain-community langchain-openai

# Embeddings + vector DB
%pip install -q sentence-transformers chromadb

# Utility
%pip install -q python-dotenv tqdm

In [ ]:
import os
import io
import gc
from tqdm import tqdm

import fitz  # PyMuPDF
import pytesseract
from PIL import Image #for iamge manipulation Resizing, Cropping, and Rotating/Format Conversion

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_openai import ChatOpenAI

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pdf_path = "/content/drive/MyDrive/medical_rag_project/bates.pdf"
VECTOR_DB_DIR = "//content/drive/MyDrive/medical_rag_project/vectordb"

BATCH_SIZE = 6

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_1684/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectordb = Chroma(
    collection_name="bates_medical_index",
    persist_directory=VECTOR_DB_DIR,
    embedding_function=embeddings
)

/tmp/ipykernel_1684/2088244663.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

In [ ]:
def process_pdf_batch(start_page, end_page):

    doc = fitz.open(pdf_path)

    collected_text = []

    for page_number in range(start_page, end_page):

        page = doc.load_page(page_number)

        text = page.get_text()

        images = page.get_images(full=True)

        for img in images:

            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            image = Image.open(io.BytesIO(image_bytes))

            ocr_text = pytesseract.image_to_string(image)

            text += "\n" + ocr_text

        collected_text.append(text)

    doc.close()

    docs = splitter.create_documents(collected_text)

    vectordb.add_documents(docs)
    vectordb.persist()

    del collected_text
    del docs
    gc.collect()
    # wihtout this all 3,Colab RAM increases each batch->crash



In [ ]:
doc = fitz.open(pdf_path)

total_pages = len(doc)

doc.close()

for i in tqdm(range(0, total_pages, BATCH_SIZE)):

    start = i
    end = min(i + BATCH_SIZE, total_pages)

    process_pdf_batch(start, end)
# If you load entire PDF(~1600 pages with images ,ocr and embbed):collab ram crashes->so batching helps to keep memory usage stable

  0%|          | 0/358 [00:00<?, ?it/s]/tmp/ipykernel_1684/4044973035.py:34: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()
100%|██████████| 358/358 [24:46<00:00,  4.15s/it]


In [ ]:
retriever = vectordb.as_retriever(search_kwargs={"k":6}) #retriever searches the vector database for relevant chunks and sent to the LLM as context.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("MODEL:", os.getenv("LLM_MODEL"))
print("KEY:", os.getenv("OPENROUTER_API_KEY"))


MODEL: meta-llama/llama-3.3-70b-instruct:free
KEY: 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


In [ ]:
load_dotenv("/content/.env")

True

In [ ]:
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)


In [ ]:
question_normalizer = ChatPromptTemplate.from_template("""
Rephrase the following question to be a standalone question using the Chat History.
Do NOT add new meaning.

Chat History:
{chat_history}

Question:
{question}

Standalone Question:
""")

answer_prompt = ChatPromptTemplate.from_template("""
You are a document-grounded medical assistant.

Follow these rules strictly:
- Answer ONLY using the provided context.
- If the context does not contain the answer say:
  "The document does not specify this information."
- Format the answer in clean Markdown.

Use this structure:

### Answer
- **Key Point 1**
- **Key Point 2**
- **Key Point 3**

### Explanation
Provide a short explanation in 2-3 sentences.

Context:
{context}

Conversation History:
{chat_history}

Question:
{question}

Answer:
""")



In [ ]:
print("🔍 Testing LLM Connectivity...")
try:
    test_val = (question_normalizer | llm | StrOutputParser()).invoke({
        "question": "What is the capital of France?",
        "chat_history": ""
    })
    print(f"✅ Standalone question check: {test_val}")
except Exception as e:
    print(f"❌ LLM Error: {e}")

🔍 Testing LLM Connectivity...
✅ Standalone question check: What is the capital of France?


In [ ]:
rag_chain = RunnablePassthrough.assign(
    standalone_question = (
        {
            "question": lambda x: x["question"],
            "chat_history": lambda x: x["chat_history"]
        }
        | question_normalizer
        | llm
        | StrOutputParser()
    )
) | {
    "context": lambda x: "\n\n".join(
        doc.page_content for doc in retriever.invoke(x["standalone_question"])
    ),
    "question": lambda x: x["standalone_question"],
    "chat_history": lambda x: x["chat_history"]
} | answer_prompt | llm | StrOutputParser()

In [ ]:
from IPython.display import display, Markdown
chat_history = []

def ask(question):

    history_text = "\n".join(chat_history)

    answer = rag_chain.invoke({
        "question": question,
        "chat_history": history_text
    })

    chat_history.append(f"User: {question}")
    chat_history.append(f"Assistant: {answer}")
    display(Markdown(answer))
    return answer

In [ ]:
ask("Patient is pregnant with dyspnea.What cardiovascular changes occur in pregnancy?")

ValidationError: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [ ]:
ask(What are the major components of a complete health history?)
### Major Components of a Complete Health History\nThe major components of a complete health history are outlined in **Box 3-2** and include:\n* **Initial information**\n* **Chief complaint(s)**\n* **History of present illness**\n* **Past medical history**\n* **Family history**\n* **Personal and social history**\n* **Review of systems**
ask("What abnormalities may appear in the inspection of the thorax and lungs?")
### Abnormalities in Thorax and Lung Inspection
The document does not specify a comprehensive list of abnormalities that may appear in the inspection of the thorax and lungs. However, it mentions a few examples of findings that may be recorded during the examination, such as:
* **Moderate kyphosis** and **increased AP diameter**
* **Decreased expansion** of the thorax
* **Paradoxical movement of the abdomen** (inward motion during inspiration), which is a sign of **diaphragmatic weakness**

These findings can indicate various respiratory or other systemic issues, but the document does not provide an exhaustive list of possible abnormalities that may be observed during the inspection of the thorax and lungs.
ask("How does pregnancy affect cardiovascular physiology?")
### Cardiovascular Changes in Pregnancy
The provided context does specify information about how pregnancy affects cardiovascular physiology. The key changes include:
* **Heart**:
  + Increased cardiac output up to 50%
  + Heart displaced left and upward
  + Exaggerated split S1
  + Hyperdynamic left ventricular (LV) function
  + Related to both increased pulse and stroke volume
  + Further augmented by almost 20% in multifetal gestations
  + Appearance of cardiomegaly on imaging
  + Systolic murmurs are common, in up to 90% of pregnant patients
* **Peripheral Vasculature**:
  + Decreased systemic vascular resistance
  + Decreased blood pressure (diastolic > systolic)
  + Decreased venous flow in the lower extremities due to compression by the gravid uterus
  + Increased venous pooling and postural hypotension
  + Increased dependent edema and varicose veins
  + Predisposes to thrombosis

  ask("What diagrams illustrate fetal positioning during pregnancy examination?")
  ### Answer\nThe document does not specify this information.

  ask("What are the *four Leopold maneuvers used to assess fetal position*?")
  ### Leopold Maneuvers
The four Leopold maneuvers used to assess fetal position are:
* **First Maneuver (Upper Fetal Pole)**: Determines what fetal part is located at the fundus.
* **Second Maneuver (Sides of the Maternal Abdomen)**: Identifies the fetal back and extremities.
* **Third Maneuver (Lower Fetal Pole and Descent into Pelvis)**: Assesses the presenting fetal part and its descent into the maternal pelvis.
* **Fourth Maneuver (Flexion of the Fetal Head)**: Evaluates the flexion or extension of the fetal head.

In [ ]:
ask(What are signs of COPD on thorax inspection?)
RAG retrieves:

barrel chest

increased AP diameter

decreased chest expansion

This helps diagnose Chronic Obstructive Pulmonary Disease.
ask("Patient is pregnant with dyspnea.What cardiovascular changes occur in pregnancy?")
Model retrieves:

cardiac output increase

venous pooling

hypotension

Doctor uses this to rule out:

pulmonary embolism

heart failure